# 06b-o — playground multifattoriale della sorgente di membrana

Questo run train-only confronta direttamente la previsione di $\Delta V$ con la previsione di una sorgente efficace applicata tramite il solve di Hines. Nella stessa matrice 2×2×2 vengono isolati feedback dello STATE e memoria locale. Oracle e controfattuali topologici non richiedono training aggiuntivo e non sono selezionabili.

Il notebook legge soltanto ruoli derivati dal train. Validation, test e fresh test restano esclusi.


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';DEFAULT_ELM_REF='abe09f40a737f5df183bd2c3801c3beefe02c323';ELM_REF=os.environ.get('HAYFLOW_ELM_REF',DEFAULT_ELM_REF)
ROOT=Path('/kaggle/working');WORKSPACE=ROOT/'hayflow_workspace';ELM_REPO=WORKSPACE/'elmneuron';WORKSPACE.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();MODULE_PATH=ELM_REPO/'src/hayflow_model/effective_membrane_source_playground.py';assert MODULE_PATH.is_file(),f'Modulo 06b-o assente nel checkout {REVISION}: {MODULE_PATH}';[sys.modules.pop(name,None) for name in tuple(sys.modules) if name=='src' or name.startswith('src.')];sys.path=[str(ELM_REPO)]+[entry for entry in sys.path if entry!=str(ELM_REPO)];importlib.invalidate_caches();print({'revision':REVISION,'requested_ref':ELM_REF,'module':str(MODULE_PATH),'module_exists':MODULE_PATH.is_file()})


## 1. Preflight minimo e immutabile

Servono soltanto il dataset composito, il risultato terminale 05t che definisce i ruoli train e l'artefatto 06b-n che autorizza la revisione. Non viene rimontata l'intera catena 06b.


In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source,materialize_nested_indexed_artifact_source
from src.hayflow_model.atomic_state_dynamics_playground import EXPECTED_05T_INDEX_SHA256
from src.hayflow_model.effective_membrane_source_playground import EXPECTED_06BN_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input');CACHE=Path('/kaggle/working/.06bo_nested_inputs')
def indexed(label,expected,env):
 override=os.environ.get(env);source=discover_indexed_artifact_source(INPUT_ROOT,expected,override=Path(override) if override else None)
 if source is None:source=materialize_nested_indexed_artifact_source(INPUT_ROOT,expected,CACHE)
 assert source is not None,f'Artefatto {label} esatto non trovato. Aggiungilo agli Input Kaggle oppure imposta {env}.'
 return source
ARTIFACT_05T_SOURCE=indexed('05t',EXPECTED_05T_INDEX_SHA256,'HAYFLOW_05T_ARTIFACT')
ARTIFACT_06BN_SOURCE=indexed('06b-n',EXPECTED_06BN_INDEX_SHA256,'HAYFLOW_06BN_ARTIFACT')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06bo_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.'
print({'05t':str(ARTIFACT_05T_SOURCE),'06b-n':str(ARTIFACT_06BN_SOURCE),'base':str(BASE_SOURCE),'topup':str(TOPUP_SOURCE)})


In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 06b-o][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880;print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})


In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model.effective_membrane_source_playground import EffectiveMembraneSourceConfig,EffectiveMembraneSourcePlayground
values=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_effective_membrane_source_playground.yml').read_text())['effective_membrane_source_playground'];config=EffectiveMembraneSourceConfig.from_mapping(values)
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_effective_membrane_source_playground');assert not OUTPUT_DIR.exists(),f'Output gia presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
session=EffectiveMembraneSourcePlayground(bundle,OUTPUT_DIR,config,ARTIFACT_05T_SOURCE,ARTIFACT_06BN_SOURCE,code_revision=REVISION);contract=session.prepare_effective_membrane_source_playground()
display({'valid':contract['valid'],'06b-n':contract['source_06bn'],'axes':contract['factorial_axes'],'arms':contract['factor_arm_count'],'parameters_per_arm':contract['trainable_parameter_count_per_arm'],'same_input':contract['same_numeric_input_tensor'],'splits':contract['state_and_outcome_splits_read']});assert contract['valid'] and contract['factor_arm_count']==8 and contract['same_numeric_input_tensor'] and not contract['validation_state_accessed']


In [ ]:
audit=session.run_exact_source_reconstruction_audit()
display({'valid':audit['valid'],'exact_error_mv':audit['maximum_authentic_reconstruction_error_mv'],'relabelled_rmse_mv':audit['median_relabelled_rmse_mv'],'no_axial_rmse_mv':audit['median_no_axial_rmse_mv'],'passive_rmse_mv':audit['median_passive_only_rmse_mv'],'teacher_oracle_selectable':audit['teacher_source_is_selection_eligible']});assert audit['valid'] and not audit['teacher_source_is_selection_eligible']


In [ ]:
training=session.train_synchronized_source_matrix()
selected={seed:{arm:row['selected_step'] for arm,row in report['selected'].items()} for seed,report in training['reports'].items()}
display({'valid':training['valid'],'arms':training['factor_arm_count'],'same_batches':training['same_minibatch_stream_within_seed'],'development_used':training['development_used_during_training'],'selected_steps':selected});assert training['valid'] and training['factor_arm_count']==8 and not training['development_used_during_training']


In [ ]:
evaluation=session.evaluate_source_matrix();final_report=session.finalize_effective_source_playground(audit,training,evaluation)
ranking=sorted(final_report['summaries'].items(),key=lambda item:item[1]['median_voltage_rmse_mv'])
compact=[{'arm':name,'rmse_8ms_mv':round(row['median_voltage_rmse_mv'],4),'gain_vs_persistence':round(row['median_gain_vs_persistence_fraction'],4),'STATE_gain':round(row['median_STATE_gain_vs_persistence_fraction'],4),'teacher_STATE_headroom':round(row['teacher_state_refresh_gain_fraction'],4),'relabelled_degradation':round(row['relabelled_topology_degradation_fraction'],4),'shuffle_degradation':round(row['spatial_shuffle_degradation_fraction'],4)} for name,row in ranking]
display(compact);display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'best':final_report['best_observed_arm'],'selected':final_report['selected_candidate'],'main_effects':final_report['factor_main_effects'],'interactions':final_report['paired_log_rmse_interactions'],'topology_signal':final_report['authentic_topology_signal_for_best_arm'],'next_step':final_report['next_step']});assert final_report['valid'] and not final_report['validation_state_accessed'] and not final_report['test_state_accessed']


## 2. Download stabile

La cella usa il metodo Blob/base64 concordato e non stampa tensori, snapshot o sequenze estese.


In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_effective_membrane_source_playground','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'download_started':filename,'size_mib':round(zip_path.stat().st_size/2**20,2)})
